In [19]:
import os
import numpy as np

# Define paths
folder1_path = "vid_txt"  # 2400-row files
folder2_path = "aud_txt"  # 640000-row files

# Load text file as numpy array
def load_txt_as_array(file_path):
    return np.loadtxt(file_path)  # Assumes 1D numerical data per file

# Get file names
folder1_files = sorted(os.listdir(folder1_path))
folder2_files = sorted(os.listdir(folder2_path))

matches = {}
used_files = set()  # Keep track of matched files from folder2

for file1 in folder1_files:
    data1 = load_txt_as_array(os.path.join(folder1_path, file1))  # (2400,)

    best_match = None
    best_corr_score = float("-inf")

    for file2 in folder2_files:
        if file2 in used_files:  # Skip already matched files
            continue

        data2 = load_txt_as_array(os.path.join(folder2_path, file2))  # (640000,)

        # Use a sliding window approach to find the best correlation
        max_corr = float("-inf")
        best_subsequence_start = 0

        for i in range(len(data2) - len(data1) + 1):  # Sliding window
            subseq2 = data2[i : i + len(data1)]  # Extract 2400-row subsequence

            # Compute cross-correlation
            correlation = np.correlate(data1, subseq2, mode="valid")[0]

            if correlation > max_corr:
                max_corr = correlation
                best_subsequence_start = i  # Save best match position

        if max_corr > best_corr_score:
            best_corr_score = max_corr
            best_match = file2

    if best_match:
        matches[file1] = best_match
        used_files.add(best_match)  # Mark as used
        print(f"Matched {file1} -> {best_match} (Cross-Corr Score: {best_corr_score:.4f})")

# Print final mappings
print("\nFinal One-to-One Matches:")
for key, value in matches.items():
    print(f"{key} -> {value}")


Matched vid_ID_1.txt -> audio_ID_14.txt (Cross-Corr Score: 26.0000)
Matched vid_ID_10.txt -> audio_ID_2.txt (Cross-Corr Score: 22.0000)
Matched vid_ID_11.txt -> audio_ID_5.txt (Cross-Corr Score: 20.0000)
Matched vid_ID_12.txt -> audio_ID_17.txt (Cross-Corr Score: 176.0000)
Matched vid_ID_13.txt -> audio_ID_22.txt (Cross-Corr Score: 20.0000)
Matched vid_ID_14.txt -> audio_ID_1.txt (Cross-Corr Score: 12.0000)
Matched vid_ID_15.txt -> audio_ID_11.txt (Cross-Corr Score: 13.0000)
Matched vid_ID_16.txt -> audio_ID_13.txt (Cross-Corr Score: 18.0000)
Matched vid_ID_17.txt -> audio_ID_10.txt (Cross-Corr Score: 12.0000)
Matched vid_ID_18.txt -> audio_ID_19.txt (Cross-Corr Score: 13.0000)
Matched vid_ID_19.txt -> audio_ID_12.txt (Cross-Corr Score: 16.0000)
Matched vid_ID_2.txt -> audio_ID_37.txt (Cross-Corr Score: 106.0000)
Matched vid_ID_20.txt -> audio_ID_15.txt (Cross-Corr Score: 15.0000)
Matched vid_ID_21.txt -> audio_ID_21.txt (Cross-Corr Score: 19.0000)
Matched vid_ID_22.txt -> audio_ID_29.

In [20]:
import pandas as pd

# Convert dictionary to DataFrame with correct extensions
df = pd.DataFrame(
    [(value.replace("audio_ID_", "audio_only_ID_").replace(".txt", ".wav"), 
      key.replace("vid_ID_", "video_only_ID_").replace(".txt", ".mp4"))
     for key, value in matches.items()],
    columns=["audio_only_ID_i", "video_only_ID_i"]
)

# Save as CSV
csv_filename = "matching_results.csv"
df.to_csv(csv_filename, index=False)

print(f"Matching results saved to {csv_filename}")


Matching results saved to matching_results.csv
